In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
project_root = os.path.abspath("")
if project_root not in sys.path:
    sys.path.append(project_root)
import sleap
from pathlib import Path
from ipywidgets import widgets
from IPython.display import display
import matplotlib.pyplot as plt
from hypnose_analysis.utils.visualization_utils import *
from sleap_utils import *

%matplotlib widget

INFO:numexpr.utils:Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [4]:
# Wrapper function to process all .slp files. 
# Params:
    # subjid: list of subjids to process (e.g., [40, 41, 42])
    # date: specific date or list of dates to process (e.g., [20251030, 20251231]) or None for all dates
    # core_nodes: list of SLEAP nodes used to calculate centroid (default uses most reliable nodes)
    # recompute: if True, recomputes and overwrites existing output files. Keep False unless SLEAP model has been updated. 
    # base_dir: optional to set to different base dir and override symlinked default path
sleap_processing = process_sleap_sessions(
    subjid=[40],
    date=[20251211],
    base_dir=None,
    core_nodes=None, 
    save_output=True, 
    recompute=True
)

Found 4 video(s) to process:
  1. 2025-12-11T14-31-20__VideoData_1904-01-16T02-00-00.predictions.slp
  2. 2025-12-11T14-31-20__VideoData_1904-01-16T03-00-00.predictions.slp
  3. 2025-12-11T14-53-02__VideoData_1904-01-16T03-00-00.predictions.slp
  4. 2025-12-11T14-53-02__VideoData_1904-01-16T04-00-00.predictions.slp

[1/4] Processing: 2025-12-11T14-31-20__VideoData_1904-01-16T02-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video1_2025-12-11T14-31-20__VideoData_1904-01-16T02-00-00.csv
    Total rows: 31880
    Frame range: 3088 to 37899

[2/4] Processing: 2025-12-11T14-31-20__VideoData_1904-01-16T03-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video2_2025-12-11T14-31-20__VideoData_1904-01-16T03-00-00.csv
    Total rows: 34836
    Frame range: 0 to 36560

[3/4] Processing: 2025-12-11T14-53-02__VideoData_1904-01-16T03-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video3_2025-12-11T14-53-02__VideoData_1904-01-16T03-00-00.csv
    Total rows: 147050
    Frame range: 0 to 1758

## Step by Step .slp file processing (use above wrapper for multiple subjids/dates)

In [13]:
# 1. Step: extract frames from .slp files and save with centroid coordinates in csv files per video

subjid = 40
date = 20251030

sleap_data = sleap_labels_and_centroid(subjid=subjid, date=date, skip_empty=True)

Found 4 video(s) to process:
  1. VideoData_1904-01-01T02-00-00.predictions.slp
  2. VideoData_1904-01-01T03-00-00.predictions.slp
  3. VideoData_1904-01-01T04-00-00.predictions.slp
  4. VideoData_1904-01-01T05-00-00.predictions.slp

[1/4] Processing: VideoData_1904-01-01T02-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video1.csv
    Total rows: 33311
    Frame range: 566 to 43849

[2/4] Processing: VideoData_1904-01-01T03-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video2.csv
    Total rows: 94856
    Frame range: 0 to 157337

[3/4] Processing: VideoData_1904-01-01T04-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video3.csv
    Total rows: 42167
    Frame range: 123904 to 182857

[4/4] Processing: VideoData_1904-01-01T05-00-00.predictions.slp
  ⚠️ No pose data found in \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\derivatives\sub-040_id-259\ses-003_date-20251030\saved_analysis_results\VideoData_1904-01-01T05-00-00.predictions.slp, skipping this file

✅ All videos proc

In [ ]:
# 2. Step: add timestamps to all video tracking csv files and combine into single dataframe
tracking_times = add_timestamps_to_sleap_tracking(subjid=40, date=20251030)

Found 3 SLEAP tracking file(s)
Found 1 experiment folder(s)
Could not create odour_led (missing output data)
  Loaded 579262 frames from experiment 0
Total: 579,262 frames from 4 video file(s)

Video files in order:
  1. VideoData_1904-01-01T02-00-00.avi
  2. VideoData_1904-01-01T03-00-00.avi
  3. VideoData_1904-01-01T04-00-00.avi
  4. VideoData_1904-01-01T05-00-00.avi
Matched sleap_tracking_video1.csv (video 1) to VideoData_1904-01-01T02-00-00.avi
Matched sleap_tracking_video2.csv (video 2) to VideoData_1904-01-01T03-00-00.avi
Matched sleap_tracking_video3.csv (video 3) to VideoData_1904-01-01T04-00-00.avi

Processing sleap_tracking_video1.csv:
  SLEAP frames: 566 - 43849 (33311 rows)
  Video local frames: 0 - 43849
  Matched 33311/33311 frames to timestamps

Processing sleap_tracking_video2.csv:
  SLEAP frames: 0 - 157337 (94856 rows)
  Video local frames: 0 - 216007
  Matched 94856/94856 frames to timestamps

Processing sleap_tracking_video3.csv:
  SLEAP frames: 123904 - 182857 (421

## Video Creation: Create an annotated video with centroid overlay and odor presentation + reward status information

In [9]:
output_video = annotate_videos_with_sleap_and_trials(
    subjid=40,
    date=20251211,
    rotate_deg=90,
    base_dir="Z:/hypnose",
    time_window=("00:00:00", "00:01:06"), 
    video_indices=[3], # which videos to annotate, can be multiple by passing [1, 2, 3] or all by passing None
    reward_display_s=2
)
# good example video quintuples: 40, 20251120, index 2, 0:10:38 to 0:11:10 (3 trials, B, A, B)

Loaded combined timestamps: 243513 frames
Odor mapping loaded successfully

Processing video 1/1 (original #3): 2025-12-11T14-53-02__VideoData_1904-01-16T03-00-00.avi
  Trimmed to window: 0 days 00:00:00 - 0 days 00:01:06 (3585 frames)
  Found 3585 frames with timestamps
  Video: 1280x1024 @ 60.0 fps, 175804 frames
  Saving to: Z:\hypnose\derivatives\sub-040_id-259\ses-035_date-20251211\saved_analysis_results\sleap_visualization_rotdeg_90_video3.mp4


  Encoding: 100%|██████████| 3917/3917 [01:01<00:00, 63.95frames/s]

  ✓ Completed!

✅ All videos processed and saved!


In [5]:
from pathlib import Path
import pandas as pd
from math import floor, ceil
from hypnose_analysis.paths import get_derivatives_root

# Given absolute clock times, find which video covers that window and compute per-video offsets
subjid = 40
date = 20251212
start_time_str = "14:42:12"  # HH:MM:SS (rounded down to this second)
end_time_str   = "14:42:55"  # HH:MM:SS (rounded up to this second)

def _fmt_hhmmss(seconds):
    seconds = max(seconds, 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

date_str = str(date)
sub_str = f"sub-{subjid:03d}"
deriv_root = get_derivatives_root()
sub_dirs = list(deriv_root.glob(f"{sub_str}_id-*"))
if not sub_dirs:
    raise FileNotFoundError(f"No subject dir for {sub_str} under {deriv_root}")
ses_dirs = list(sub_dirs[0].glob(f"ses-*_date-{date_str}"))
if not ses_dirs:
    raise FileNotFoundError(f"No session dir for date {date_str}")
results_dir = ses_dirs[0] / "saved_analysis_results"

combined_files = list(results_dir.glob("*_combined_sleap_tracking_timestamps.csv"))
if not combined_files:
    raise FileNotFoundError("No combined timestamps CSV found; run add_timestamps_to_sleap_tracking first")
combined_path = combined_files[0]

df = pd.read_csv(combined_path)
if "time" not in df.columns or "video_file" not in df.columns:
    raise ValueError("Combined CSV missing required columns 'time' or 'video_file'")

# Normalize times to naive for comparison
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df["time_naive"] = df["time"].dt.tz_localize(None)
start_dt = pd.to_datetime(f"{date_str} {start_time_str}", errors="coerce")
end_dt = pd.to_datetime(f"{date_str} {end_time_str}", errors="coerce")
if pd.isna(start_dt) or pd.isna(end_dt) or end_dt <= start_dt:
    raise ValueError("Invalid start/end times")

# Order videos by first appearance in the combined file
video_order = []
seen = set()
for vf in df["video_file"]:
    if vf not in seen:
        video_order.append(vf)
        seen.add(vf)
video_index_map = {vf: idx + 1 for idx, vf in enumerate(video_order)}  # 1-based

summary = []
for vf, sub in df.groupby("video_file"):
    t_min = sub["time_naive"].min()
    t_max = sub["time_naive"].max()
    summary.append((vf, t_min, t_max))

# Find the video that contains the start time
matches = [item for item in summary if item[1] <= start_dt <= item[2]]
if not matches:
    print("No video covers the requested start time.")
else:
    vf, t_min, t_max = matches[0]
    idx = video_index_map.get(vf, None)
    # Round start down to whole second, end up to whole second (but not past video end)
    start_offset_raw = (start_dt - t_min).total_seconds()
    end_offset_raw = (min(end_dt, t_max) - t_min).total_seconds()
    start_offset = floor(start_offset_raw)
    end_offset = ceil(end_offset_raw)
    print(f"Video file: {vf}")
    if idx is not None:
        print(f"Video index (1-based): {idx}")
    print(f"Time window for annotate_videos_with_sleap_and_trials: (\"{_fmt_hhmmss(start_offset)}\", \"{_fmt_hhmmss(end_offset)}\")")
    print(f"Video covers {t_min} to {t_max} (clock time)")
    if end_dt > t_max:
        over = (end_dt - t_max).total_seconds()
        print(f"Warning: requested end extends {over:.2f}s past this video; window clipped to video end")

Video file: VideoData_1904-01-17T03-00-00.avi
Video index (1-based): 2
Time window for annotate_videos_with_sleap_and_trials: ("00:00:22", "00:01:06")
Video covers 2025-12-12 14:41:49.006432 to 2025-12-12 14:57:32.156320 (clock time)


In [13]:
import cv2
from collections import defaultdict
from hypnose_analysis.paths import get_data_root

# Diagnose per-video coverage vs combined timestamps (checks for leading gaps)
subjid = 40
date = 20251211
video_number = 2  # 1-based across all behav/*/VideoData/*.avi in sorted order

raw_root = get_data_root() / "rawdata"
deriv_root = get_derivatives_root()

# Locate session in rawdata
sub_dir = next(iter(sorted(raw_root.glob(f"sub-{subjid:03d}_id-*"))))
ses_dir = next(iter(sorted(sub_dir.glob(f"ses-*_date-{date}"))))
behav_dirs = sorted(ses_dir.glob("behav/*"))
video_files_all = []
for behav_dir in behav_dirs:
    video_files_all.extend(sorted(behav_dir.glob("VideoData/*.avi")))
if not video_files_all:
    raise FileNotFoundError("No VideoData .avi files found under rawdata")
if video_number < 1 or video_number > len(video_files_all):
    raise ValueError(f"video_number {video_number} out of range (1..{len(video_files_all)})")

target_video = video_files_all[video_number - 1]
video_name = target_video.name
print(f"Using raw video #{video_number}: {target_video}")

# Locate matching results_dir (derivatives) for timestamps
sub_dir_deriv = next(iter(sorted(deriv_root.glob(f"sub-{subjid:03d}_id-*"))))
ses_dir_deriv = next(iter(sorted(sub_dir_deriv.glob(f"ses-*_date-{date}"))))
results_dir = ses_dir_deriv / "saved_analysis_results"

combined_files = list(results_dir.glob("*_combined_sleap_tracking_timestamps.csv"))
if not combined_files:
    raise FileNotFoundError("No combined timestamps CSV found")
combined_path = combined_files[0]
combined_df = pd.read_csv(combined_path)
combined_df["time"] = pd.to_datetime(combined_df["time"], errors="coerce")
combined_df["time_naive"] = combined_df["time"].dt.tz_localize(None)

# Stats from combined timestamps for this video (match by filename)
df_video = combined_df[combined_df["video_file"] == video_name].copy()
if df_video.empty:
    raise ValueError(f"No entries for {video_name} in combined timestamps")

frame_col = "local_frame" if "local_frame" in df_video.columns else ("frame" if "frame" in df_video.columns else None)
frame_min = df_video[frame_col].min() if frame_col else None
frame_max = df_video[frame_col].max() if frame_col else None

t_min = df_video["time_naive"].min()
t_max = df_video["time_naive"].max()
span_s = (t_max - t_min).total_seconds() if pd.notna(t_min) and pd.notna(t_max) else float("nan")

# Video duration from metadata
cap = cv2.VideoCapture(str(target_video))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
duration_s = n_frames / fps if fps > 0 else float('nan')
cap.release()

print(f"Video file: {video_name}")
print(f"Frames (metadata): {n_frames:.0f} @ {fps:.2f} fps -> duration {duration_s:.2f}s")
print(f"Combined timestamps frames: {len(df_video)}")
print(f"frame col used: {frame_col}")
print(f"frame min/max in combined: {frame_min} / {frame_max}")
if frame_min is not None and fps > 0:
    missing_front_s = frame_min / fps
    missing_back_s = max(0.0, (n_frames - 1 - frame_max) / fps)
    print(f"Approx missing at start: {missing_front_s:.2f}s; at end: {missing_back_s:.2f}s")
print(f"Time span in combined: {t_min} to {t_max} (naive)")
print(f"Time span seconds (naive): {span_s:.2f}s")

expected_frames_from_combined = frame_max - frame_min + 1 if frame_min is not None and frame_max is not None else None
if expected_frames_from_combined and fps > 0:
    approx_duration_from_combined = expected_frames_from_combined / fps
    print(f"Duration from combined frames @ metadata fps: {approx_duration_from_combined:.2f}s")
if expected_frames_from_combined and n_frames > 0:
    ratio = expected_frames_from_combined / n_frames
    if ratio < 0.95 or ratio > 1.05:
        print(f"WARNING: combined frame count {expected_frames_from_combined} vs cv2 metadata {n_frames:.0f} ({ratio:.2f}x). Metadata may be inaccurate.")
if span_s > 0 and frame_min is not None and frame_max is not None:
    est_fps_from_time = (frame_max - frame_min) / span_s
    print(f"FPS implied by timestamps: {est_fps_from_time:.2f}")

# Optional: show sleap_tracking_videoN frame range
sleap_csv = results_dir / f"sleap_tracking_video{video_number}.csv"
if sleap_csv.exists():
    df_sleap = pd.read_csv(sleap_csv, nrows=200000)
    if "frame" in df_sleap:
        print(f"SLEAP CSV frames: {df_sleap['frame'].min()} to {df_sleap['frame'].max()} (rows: {len(df_sleap)})")
else:
    print(f"SLEAP CSV not found: {sleap_csv.name}")

Using raw video #2: C:\Users\HarrisLab\Desktop\Repos\hypnose\hypnose-analysis\data\rawdata\sub-040_id-259\ses-035_date-20251211\behav\2025-12-11T14-31-20\VideoData\VideoData_1904-01-16T03-00-00.avi
Video file: VideoData_1904-01-16T03-00-00.avi
Frames (metadata): 36561 @ 60.00 fps -> duration 609.35s
Combined timestamps frames: 181914
frame col used: frame
frame min/max in combined: 0 / 175803
Approx missing at start: 0.00s; at end: 0.00s
Time span in combined: 2025-12-11 14:41:52.000480 to 2025-12-11 15:41:51.995296 (naive)
Time span seconds (naive): 3599.99s
Duration from combined frames @ metadata fps: 2930.07s
FPS implied by timestamps: 48.83
SLEAP CSV frames: 0 to 175803 (rows: 147050)


In [16]:
import subprocess
import json
from fractions import Fraction
import cv2
from hypnose_analysis.paths import get_data_root

# Cross-check video metadata with ffprobe (requires ffprobe/ffmpeg in PATH)
subjid = 40
date = 20251211
video_number = 2

raw_root = get_data_root() / "rawdata"
sub_dir = next(iter(sorted(raw_root.glob(f"sub-{subjid:03d}_id-*"))))
ses_dir = next(iter(sorted(sub_dir.glob(f"ses-*_date-{date}"))))
behav_dirs = sorted(ses_dir.glob("behav/*"))
video_files_all = []
for behav_dir in behav_dirs:
    video_files_all.extend(sorted(behav_dir.glob("VideoData/*.avi")))
if not video_files_all:
    raise FileNotFoundError("No VideoData .avi files found under rawdata")
if video_number < 1 or video_number > len(video_files_all):
    raise ValueError(f"video_number {video_number} out of range (1..{len(video_files_all)})")
target_video = video_files_all[video_number - 1]
print(f"Checking with ffprobe: {target_video}")

def _parse_fps(val):
    try:
        frac = Fraction(val)
        if frac.denominator == 0:
            return None
        return float(frac)
    except Exception:
        return None

# ffprobe basic stream info
cmd = [
    "ffprobe","-v","error",
    "-select_streams","v:0",
    "-show_entries","stream=nb_frames,r_frame_rate,avg_frame_rate,duration",
    "-of","json",
    str(target_video)
]
try:
    out = subprocess.check_output(cmd, stderr=subprocess.STDOUT)
    info = json.loads(out.decode("utf-8"))
    stream = info.get("streams", [{}])[0] if info.get("streams") else {}
    nb_frames = stream.get("nb_frames")
    r_fps_raw = stream.get("r_frame_rate")
    avg_fps_raw = stream.get("avg_frame_rate")
    r_fps = _parse_fps(r_fps_raw) if r_fps_raw else None
    avg_fps = _parse_fps(avg_fps_raw) if avg_fps_raw else None
    duration = stream.get("duration")
    print(f"ffprobe nb_frames: {nb_frames}")
    if r_fps:
        print(f"ffprobe r_frame_rate: {r_fps_raw} (~{r_fps:.3f} fps)")
    if avg_fps:
        print(f"ffprobe avg_frame_rate: {avg_fps_raw} (~{avg_fps:.3f} fps)")
    if duration:
        print(f"ffprobe duration (stream): {duration}")
except FileNotFoundError:
    print("ffprobe not found in PATH; install ffmpeg to use this check.")
except subprocess.CalledProcessError as e:
    print("ffprobe failed:")
    print(e.output.decode("utf-8", errors="ignore"))

# cv2 fallback for comparison
cap = cv2.VideoCapture(str(target_video))
fps_cv = cap.get(cv2.CAP_PROP_FPS)
n_frames_cv = cap.get(cv2.CAP_PROP_FRAME_COUNT)
cap.release()
print(f"cv2 FPS: {fps_cv:.3f}, cv2 n_frames: {n_frames_cv:.0f}")

Checking with ffprobe: C:\Users\HarrisLab\Desktop\Repos\hypnose\hypnose-analysis\data\rawdata\sub-040_id-259\ses-035_date-20251211\behav\2025-12-11T14-31-20\VideoData\VideoData_1904-01-16T03-00-00.avi
ffprobe nb_frames: 36561
ffprobe r_frame_rate: 60/1 (~60.000 fps)
ffprobe avg_frame_rate: 60/1 (~60.000 fps)
ffprobe duration (stream): 609.350000
cv2 FPS: 60.000, cv2 n_frames: 36561


In [6]:
import pandas as pd
import cv2
from hypnose_analysis.paths import get_derivatives_root, get_data_root

# Summarize raw videos vs combined timestamps and sleap csvs
subjid = 40
date = 20251211

raw_root = get_data_root() / "rawdata"
sub_dir = next(iter(sorted(raw_root.glob(f"sub-{subjid:03d}_id-*"))))
ses_dir = next(iter(sorted(sub_dir.glob(f"ses-*_date-{date}"))))
behav_dirs = sorted(ses_dir.glob("behav/*"))
video_files_all = []
for behav_dir in behav_dirs:
    video_files_all.extend(sorted(behav_dir.glob("VideoData/*.avi")))
if not video_files_all:
    raise FileNotFoundError("No VideoData .avi files found under rawdata")

def _vid_meta(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    dur = n_frames / fps if fps > 0 else float('nan')
    return fps, n_frames, dur

raw_rows = []
for idx, vf in enumerate(video_files_all, 1):
    fps, n_frames, dur = _vid_meta(vf)
    raw_rows.append({"video_idx": idx, "video_file": vf.name, "fps_cv2": fps, "n_frames_cv2": n_frames, "dur_s_cv2": dur})
raw_df = pd.DataFrame(raw_rows)

# Combined summary
deriv_root = get_derivatives_root()
sub_dir_deriv = next(iter(sorted(deriv_root.glob(f"sub-{subjid:03d}_id-*"))))
ses_dir_deriv = next(iter(sorted(sub_dir_deriv.glob(f"ses-*_date-{date}"))))
results_dir = ses_dir_deriv / "saved_analysis_results"
combined_files = list(results_dir.glob("*_combined_sleap_tracking_timestamps.csv"))
if not combined_files:
    raise FileNotFoundError("No combined timestamps CSV found")
combined_path = combined_files[0]
combined_df = pd.read_csv(combined_path)
if "time" in combined_df:
    combined_df["time"] = pd.to_datetime(combined_df["time"], errors="coerce")
    combined_df["time_naive"] = combined_df["time"].dt.tz_localize(None)
comb_rows = []
for vf, sub in combined_df.groupby("video_file"):
    frame_col = "local_frame" if "local_frame" in sub.columns else ("frame" if "frame" in sub.columns else None)
    fmin = sub[frame_col].min() if frame_col else None
    fmax = sub[frame_col].max() if frame_col else None
    comb_rows.append({"video_file": vf, "rows_combined": len(sub), "frame_min": fmin, "frame_max": fmax})
comb_df = pd.DataFrame(comb_rows)

# Sleap per-index summary (frame range) using tracking csvs in results_dir
sleap_rows = []
for idx in range(1, len(video_files_all) + 1):
    sleap_csv = results_dir / f"sleap_tracking_video{idx}.csv"
    if sleap_csv.exists():
        df_head = pd.read_csv(sleap_csv, nrows=0)
        usecols = [c for c in ["frame", "local_frame"] if c in df_head.columns]
        df_s = pd.read_csv(sleap_csv, usecols=usecols) if usecols else pd.DataFrame()
        col = "local_frame" if "local_frame" in df_s.columns else ("frame" if "frame" in df_s.columns else None)
        fmin = df_s[col].min() if col else None
        fmax = df_s[col].max() if col else None
        rows = len(df_s)
    else:
        fmin = fmax = rows = None
    sleap_rows.append({"video_idx": idx, "sleap_csv": sleap_csv.name, "frame_min": fmin, "frame_max": fmax, "rows": rows})
sleap_df = pd.DataFrame(sleap_rows)

print("Raw videos (cv2 metadata):")
print(raw_df)
print("\nCombined timestamps by video_file:")
print(comb_df)
print("\nSLEAP tracking CSVs by index:")
print(sleap_df)

print("\nJoin raw vs combined on filename (outer):")
joined = raw_df.merge(comb_df, on="video_file", how="outer")
print(joined)

print("\nNote: If combined or sleap rows far exceed cv2 counts, mapping/video association is likely wrong.")

Raw videos (cv2 metadata):
   video_idx                           video_file  fps_cv2  n_frames_cv2  \
0          1    VideoData_1904-01-16T02-00-00.avi     60.0       37900.0   
1          2    VideoData_1904-01-16T03-00-00.avi     60.0       36561.0   
2          3  ._VideoData_1904-01-16T03-00-00.avi      0.0           0.0   
3          4    VideoData_1904-01-16T03-00-00.avi     60.0      175804.0   
4          5    VideoData_1904-01-16T04-00-00.avi     60.0       41337.0   

     dur_s_cv2  
0   631.666667  
1   609.350000  
2          NaN  
3  2930.066667  
4   688.950000  

Combined timestamps by video_file:
                                          video_file  rows_combined  \
0  2025-12-11T14-31-20__VideoData_1904-01-16T02-0...          31880   
1  2025-12-11T14-31-20__VideoData_1904-01-16T03-0...          34836   
2  2025-12-11T14-53-02__VideoData_1904-01-16T03-0...         147050   
3  2025-12-11T14-53-02__VideoData_1904-01-16T04-0...          29747   

   frame_min  frame_ma